# House Price Prediction – 10-Fold Cross Validation

**Name:** Muhammad Hassan Yousaf  
**Roll Number:** FA24-BSE-082  
**Course:** Machine Learning (CSC-461) – Assignment 3, Task 2  

**Deployment Link:** [https://house-price-hassan.streamlit.app](https://house-price-hassan.streamlit.app)  
**GitHub:** [https://github.com/MHassan0000/ML-Prediction-Models](https://github.com/MHassan0000/ML-Prediction-Models)

---

**Problem:** Predict the median house value in California using 8 features like income, house age, rooms, location etc.

**Dataset:** California Housing (20,640 instances, 8 features)  
**Evaluation:** 10-Fold Cross Validation using RMSE

**5 Algorithms (different from Car Price Prediction):**
1. Ridge Regression
2. Lasso Regression
3. ElasticNet
4. LinearSVR
5. XGBoost

## Step 1: Import Libraries

In [ ]:
import os
import pickle
import time
import numpy as np
import pandas as pd
import sklearn
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator

from sklearn.base import clone
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.svm import LinearSVR

try:
    import xgboost as xgb
    from xgboost import XGBRegressor
except ImportError:
    !pip install xgboost -q
    import xgboost as xgb
    from xgboost import XGBRegressor

from tqdm import tqdm

pd.set_option("display.float_format", lambda v: f"{v:,.4f}")

print(f"NumPy: {np.__version__}")
print(f"Pandas: {pd.__version__}")
print(f"Scikit-learn: {sklearn.__version__}")
print(f"XGBoost: {xgb.__version__}")

## Step 2: Load Dataset

Upload `housing.csv` when prompted (on Google Colab).

In [ ]:
DATA_FILE = "housing.csv"

if not os.path.isfile(DATA_FILE):
    try:
        from google.colab import files as _colab_files
        print("Please upload housing.csv ...")
        uploaded = _colab_files.upload()
    except ImportError:
        from sklearn.datasets import fetch_california_housing
        data = fetch_california_housing(as_frame=True)
        data.frame.to_csv(DATA_FILE, index=False)
        print(f"Loaded from sklearn: {len(data.frame)} rows")
else:
    print(f"{DATA_FILE} already available")

df = pd.read_csv(DATA_FILE)
print(f"Loaded: {df.shape[0]} rows, {df.shape[1]} columns")
df.head()

## Step 3: Understand the Data

In [ ]:
print("Shape:", df.shape)
print("\nData types:")
print(df.dtypes)
print("\nMissing values:", df.isnull().sum().sum())
print()
df.describe()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(df['MedHouseVal'], bins=50, color='steelblue', edgecolor='white')
axes[0].set_title('Distribution of House Values')
axes[0].set_xlabel('Median House Value (×$100k)')
axes[0].set_ylabel('Count')

axes[1].boxplot(df['MedHouseVal'], vert=True, patch_artist=True,
                boxprops=dict(facecolor='steelblue', alpha=0.7))
axes[1].set_title('Box Plot')
axes[1].set_ylabel('Median House Value (×$100k)')

plt.tight_layout()
plt.show()

In [ ]:
# correlation heatmap
fig, ax = plt.subplots(figsize=(9, 7))
corr = df.corr()
im = ax.imshow(corr, cmap='RdBu_r', vmin=-1, vmax=1)
ax.set_xticks(range(len(corr.columns)))
ax.set_yticks(range(len(corr.columns)))
ax.set_xticklabels(corr.columns, rotation=45, ha='right')
ax.set_yticklabels(corr.columns)

for i in range(len(corr)):
    for j in range(len(corr)):
        ax.text(j, i, f'{corr.iloc[i,j]:.2f}', ha='center', va='center', fontsize=8)

plt.colorbar(im)
ax.set_title('Correlation Heatmap')
plt.tight_layout()
plt.show()

## Step 4: Pre-processing

In [ ]:
df = df.drop_duplicates()
print(f"After removing duplicates: {len(df)} rows")

TARGET = "MedHouseVal"
FEATURES = [c for c in df.columns if c != TARGET]

X = df[FEATURES]
y = df[TARGET]

print(f"\nFeatures: {FEATURES}")
print(f"X shape: {X.shape}, y shape: {y.shape}")
print(f"Target range: {y.min():.2f} to {y.max():.2f} (×$100k)")

## Step 5: Pre-processing Pipeline

Since all features are numerical, we just need imputation and scaling.

In [ ]:
preprocessor = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])
print("Pipeline: MedianImputer -> StandardScaler")

## Step 6: Define 5 Algorithms

These are all different from the ones used in Car Price Prediction (which used Linear Regression, KNN, Decision Tree, Random Forest, Gradient Boosting).

In [ ]:
ALGORITHMS = {
    "Ridge Regression": Ridge(alpha=1.0),
    "Lasso Regression": Lasso(alpha=0.001, max_iter=10000),
    "ElasticNet":       ElasticNet(alpha=0.001, l1_ratio=0.5, max_iter=10000),
    "LinearSVR":        LinearSVR(C=1.0, epsilon=0.0, max_iter=5000, random_state=42),
    "XGBoost":          XGBRegressor(n_estimators=300, max_depth=6, learning_rate=0.1,
                                     subsample=0.8, colsample_bytree=0.8,
                                     random_state=42, verbosity=0),
}

for i, name in enumerate(ALGORITHMS, 1):
    print(f"  {i}. {name}")

## Step 7: Set Up 10-Fold Cross Validation

In [ ]:
K = 10
kf = KFold(n_splits=K, shuffle=True, random_state=42)

print(f"K = {K}")
print(f"Total samples: {len(X)}")
print(f"~{len(X)//K} per test fold, ~{len(X) - len(X)//K} for training")

print(f"\n{'Fold':>4}  {'Train':>8}  {'Test':>6}")
for i, (tr, te) in enumerate(kf.split(X), 1):
    print(f"{i:>4}  {len(tr):>8}  {len(te):>6}")

## Step 8: Run Cross Validation

In [ ]:
results = {}
total = len(ALGORITHMS) * K
pbar = tqdm(total=total, desc="Training", unit="fit")

for algo_name, algo in ALGORITHMS.items():
    rmses = []
    for train_idx, test_idx in kf.split(X):
        X_tr, X_te = X.iloc[train_idx], X.iloc[test_idx]
        y_tr, y_te = y.iloc[train_idx], y.iloc[test_idx]

        pipe = clone(preprocessor)
        X_tr_s = pipe.fit_transform(X_tr)
        X_te_s = pipe.transform(X_te)

        model = clone(algo)
        model.fit(X_tr_s, y_tr)

        pred = model.predict(X_te_s)
        rmse = np.sqrt(mean_squared_error(y_te, pred))
        rmses.append(rmse)
        pbar.update(1)

    results[algo_name] = rmses

pbar.close()
print("Done!")

## Step 9: Compare Results

### Per-Fold RMSE

In [ ]:
fold_df = pd.DataFrame(results, index=[f"Fold {i}" for i in range(1, K+1)])
fold_df

### Summary Table

In [ ]:
summary = pd.DataFrame({
    "Mean RMSE": {n: np.mean(v) for n, v in results.items()},
    "Std RMSE":  {n: np.std(v)  for n, v in results.items()},
    "Min RMSE":  {n: np.min(v)  for n, v in results.items()},
    "Max RMSE":  {n: np.max(v)  for n, v in results.items()},
}).sort_values("Mean RMSE")

print("Sorted by Mean RMSE (lower = better):")
summary

### Charts

In [ ]:
names = summary.index.tolist()
colors = ['#2ecc71', '#3498db', '#9b59b6', '#e67e22', '#e74c3c']

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# bar chart
m = [summary.loc[n, 'Mean RMSE'] for n in names]
s = [summary.loc[n, 'Std RMSE'] for n in names]
bars = axes[0].barh(names, m, xerr=s, color=colors, edgecolor='white', capsize=4)
axes[0].set_xlabel('Mean RMSE')
axes[0].set_title('Mean RMSE (lower = better)')
axes[0].invert_yaxis()
for bar, val in zip(bars, m):
    axes[0].text(val + 0.01, bar.get_y() + bar.get_height()/2, f'{val:.4f}', va='center')

# box plot
bp = axes[1].boxplot([results[n] for n in names], labels=names, vert=True, patch_artist=True)
for patch, c in zip(bp['boxes'], colors):
    patch.set_facecolor(c)
    patch.set_alpha(0.7)
axes[1].set_ylabel('RMSE')
axes[1].set_title('RMSE Distribution')
axes[1].tick_params(axis='x', rotation=25)

# line plot
for n, c in zip(names, colors):
    axes[2].plot(range(1, K+1), results[n], 'o-', label=n, color=c, linewidth=2)
axes[2].set_xlabel('Fold')
axes[2].set_ylabel('RMSE')
axes[2].set_title('RMSE per Fold')
axes[2].legend(fontsize=8)
axes[2].xaxis.set_major_locator(MaxNLocator(integer=True))

plt.tight_layout()
plt.show()

### Observations

In [ ]:
best = summary.index[0]
worst = summary.index[-1]

print(f"Best:  {best} (RMSE = {summary.loc[best, 'Mean RMSE']:.4f}, ~${summary.loc[best, 'Mean RMSE']*100000:,.0f} error)")
print(f"Worst: {worst} (RMSE = {summary.loc[worst, 'Mean RMSE']:.4f}, ~${summary.loc[worst, 'Mean RMSE']*100000:,.0f} error)")
print(f"\n-> {best} clearly wins. Selecting it for final model.")

## Step 10: Train Final Model

In [ ]:
BEST_NAME = summary.index[0]
print(f"Training {BEST_NAME} on full dataset...")

# fit preprocessor on all data
final_pipe = clone(preprocessor)
X_all = final_pipe.fit_transform(X)

# train
final_model = clone(ALGORITHMS[BEST_NAME])
final_model.fit(X_all, y)

# sanity check
train_rmse = np.sqrt(mean_squared_error(y, final_model.predict(X_all)))
print(f"Training RMSE: {train_rmse:.4f}")

### Save Model

In [ ]:
with open("house_price_best_model.pkl", "wb") as f:
    pickle.dump(final_model, f)

with open("house_price_preprocessor.pkl", "wb") as f:
    pickle.dump(final_pipe, f)

with open("house_price_features.pkl", "wb") as f:
    pickle.dump(FEATURES, f)

print("Model, preprocessor and features saved.")

## Step 11: Test Prediction

In [ ]:
# sample input - first row of dataset
sample = {
    "MedInc": 8.3252, "HouseAge": 41.0, "AveRooms": 6.984,
    "AveBedrms": 1.024, "Population": 322.0, "AveOccup": 2.556,
    "Latitude": 37.88, "Longitude": -122.23,
}

# load and predict
with open("house_price_best_model.pkl", "rb") as f:
    model = pickle.load(f)
with open("house_price_preprocessor.pkl", "rb") as f:
    prep = pickle.load(f)

inp = pd.DataFrame([sample])
pred = model.predict(prep.transform(inp))[0]

print(f"Input: {sample}")
print(f"Predicted: ${pred * 100000:,.0f}")
print(f"Actual:    $452,600")
print(f"Error:     ${abs(pred - 4.526) * 100000:,.0f} ({abs(pred - 4.526)/4.526*100:.1f}%)")

## Step 12: Feedback

The model will be deployed as a Streamlit web app where users can input house features and get price predictions. User feedback can be collected to retrain and improve the model in the future.

In [ ]:
print("Model is ready for deployment!")
print(f"Best algorithm: {BEST_NAME}")
print(f"CV RMSE: {summary.loc[BEST_NAME, 'Mean RMSE']:.4f} (~${summary.loc[BEST_NAME, 'Mean RMSE']*100000:,.0f})")
print(f"Files saved: house_price_best_model.pkl, house_price_preprocessor.pkl")